# Step 10a: every detector on real Sentinel-2, in one table

Steps 6, 8 and 9 each scored some of the detectors on the seven Finnish test scenes, with slightly different settings. This scores all five together, the same way, so the write-up has one baseline table:

| Name | What it is | Trained on |
|---|---|---|
| `stock` | YOLO11s as Ultralytics ships it, only its `boat` class counted | COCO photos, no satellite imagery |
| `high_res` | the detector of step 5 | ShipRSImageNet at 0.3 to 1 m |
| `at_10m` | the detector of step 7c | ShipRSImageNet resampled to 10 m |
| `finetuned` | the detector of step 9 | real Sentinel-2, the Finnish training scenes |
| `finnish_reference` | the published model of the Finnish Environment Institute | real Sentinel-2 |

`stock` against `finetuned` is the same architecture and starting point, with and without fine-tuning on Sentinel-2. `finetuned` against `finnish_reference` compares your run with the published one; both are YOLO11s. The `Dock` class of the two ShipRSImageNet detectors is ignored.

Besides the usual threshold of 0.25, every detector is also scored at stricter thresholds, and the report lists how many vessels each finds while a given share of its detections are hits (50, 60, 70%). That compares the detectors at equal precision, which step 9 showed is needed: the fine-tuned model finds more vessels at 0.25 but also raises more false alarms.

Inference only. The scenes are on Drive from steps 6 and 9.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
BRANCH = 'main'

RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'
S2_DIR = '/content/drive/MyDrive/5.Projects/Data/sentinel2_finland'

## Copernicus login (only used if a scene has to be downloaded again)

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['CDSE_USERNAME'] = userdata.get('CDSE_USERNAME')
    os.environ['CDSE_PASSWORD'] = userdata.get('CDSE_PASSWORD')
    print('Login found')
except Exception as error:
    print('No Copernicus login available, continuing with the scenes on Drive:', type(error).__name__)

## Code

In [ ]:
import os, subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn, matplotlib, pyyaml and opencv
%pip install -q --no-deps -e /content/vessel-watch
%pip install -q ultralytics
%cd /content/vessel-watch
!git log --oneline -1

os.environ['VESSELWATCH_RUNS'] = RUNS_DIR

## The fine-tuned detector on its own test chips

Written by step 9 at the end of training: precision, recall and mAP on the chips of the seven test scenes.

In [ ]:
!cat "$RUNS_DIR/yolo11s_finland/metrics_test.json"

## Run

In [ ]:
!python -m vesselwatch.detect.finland --name step_10 --data-dir "$S2_DIR" --detector stock=yolo11s.pt --keep stock=boat --detector high_res="$RUNS_DIR/yolo11s_obb/weights/best.pt" --detector at_10m="$RUNS_DIR/yolo11s_obb_10m/weights/best.pt" --detector finetuned="$RUNS_DIR/yolo11s_finland/weights/best.pt" --reference --drop-class Dock

## Examples

Labelled vessels in green, each detector's boxes in its own colour. Saved in `RUNS_DIR/sentinel2_step_10_examples/`.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in sorted(Path(RUNS_DIR, 'sentinel2_step_10_examples').glob('*.png'))[:6]:
    print(path.name)
    display(Image(filename=str(path)))